In [26]:
import torch 
import soundfile as sf
from qwen_tts import Qwen3TTSModel 

In [27]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

사용 디바이스: cuda:0


# TTS

## 1) 모델 불러오기

In [4]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",  # 가장 가벼운 모델부터
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",  # Windows는 sdpa 고정
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
import soundfile as sf
# 한국어 테스트
wavs, sr = model.generate_custom_voice(
    text="안녕하세요, 저는 인공지능 스피커입니다.",
    language="Korean",
    speaker="sohee",  # 목록 확인 후 한국어 지원 화자로 변경
)


sf.write("./audios/qwen3_test.wav", wavs[0], sr)
print("✅ test.wav 생성 완료!")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ test.wav 생성 완료!


# 보이스 클론

In [28]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",  # Base 모델
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [29]:

import time


ref_audio = "./audios/my_voice.mp3"
ref_text  = "안녕하세요. 개그우먼 이수지입니다. 반갑습니다."
emotion_label = "Scary" # 비전 모델에서 넘어온 값
start_time = time.time()
wavs, sr = model.generate_voice_clone(
    text = "이제 그만 책을 읽어요",
    language="Korean",
    ref_audio=ref_audio,
    x_vector_only_mode=True
)
end_time = time.time()
print(f"GPU 생성 소요 시간: {end_time - start_time:.2f}초")

sf.write("./audios/output_voice_clone.wav", wavs[0], sr)


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GPU 생성 소요 시간: 5.06초


In [ ]:
#cpu로 돌려보기
# 모델이 이미 생성되어 있다면
model.to("cpu")

# 이후 생성 코드 실행
ref_audio = "./audios/my_voice.mp3"
ref_text  = "안녕하세요. 개그우먼 이수지입니다. 반갑습니다."

# 시간 측정을 위해 라이브러리 추가
import time
start_time = time.time()

wavs, sr = model.generate_voice_clone(
    text = "떡 하나주면 안 잡아먹지!!!! 으르렁!!!!",
    language="Korean",
    ref_audio=ref_audio,
    ref_text=ref_text,
)

end_time = time.time()
print(f"CPU 생성 소요 시간: {end_time - start_time:.2f}초")

sf.write("./audios/output_voice_clone.wav", wavs[0], sr)

=> 아쉽게도 지시문을 text에 끼워넣는 방식은 모두 실패

# Base 모델 뜯어보기

## 1) 보이스 클론 함수 뜯어보기

In [20]:
# generate_voice_clone 소스 확인
import inspect
print(inspect.getsource(model.generate_voice_clone))

#input_texts = [self._build_assistant_text(t) for t in texts]
# talker_codes_list, _ = self.model.generate(
#     input_ids=input_ids,
#     ref_ids=ref_ids,
#     voice_clone_prompt=voice_clone_prompt_dict,
#     languages=languages,
#     non_streaming_mode=non_streaming_mode,
#     **gen_kwargs,
# )


    @torch.no_grad()
    def generate_voice_clone(
        self,
        text: Union[str, List[str]],
        language: Union[str, List[str]] = None,
        ref_audio: Optional[Union[AudioLike, List[AudioLike]]] = None,
        ref_text: Optional[Union[str, List[Optional[str]]]] = None,
        x_vector_only_mode: Union[bool, List[bool]] = False,
        voice_clone_prompt: Optional[Union[Dict[str, Any], List[VoiceClonePromptItem]]] = None,
        non_streaming_mode: bool = False,
        **kwargs,
    ) -> Tuple[List[np.ndarray], int]:
        """
        Voice clone speech using the Base model.

        You can provide either:
          - (ref_audio, ref_text, x_vector_only_mode) and let this method build the prompt, OR
          - `VoiceClonePromptItem` returned by `create_voice_clone_prompt`, OR
          - a list of `VoiceClonePromptItem` returned by `create_voice_clone_prompt`.
        
        `ref_audio` Supported forms:
        - str: wav path / URL / base64 audio string
   

In [21]:
print(inspect.getsource(model._build_assistant_text))

    def _build_assistant_text(self, text: str) -> str:
        return f"<|im_start|>assistant\n{text}<|im_end|>\n<|im_start|>assistant\n"



In [22]:
print(inspect.getsource(model._build_ref_text))


    def _build_ref_text(self, text: str) -> str:
        return f"<|im_start|>assistant\n{text}<|im_end|>\n"



In [23]:
def patched_build_assistant_text(text: str, instruction: str = "") -> str:
    if instruction:
        return (
            f"<|im_start|>user\n{instruction}<|im_end|>\n"
            f"<|im_start|>assistant\n{text}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    else:
        return f"<|im_start|>assistant\n{text}<|im_end|>\n<|im_start|>assistant\n"

# 원하는 감정을 instruction에 넣어서 호출
import types

emotion = "Speak in a sad tone"

model._build_assistant_text = lambda text: patched_build_assistant_text(text, instruction=emotion)

# 그냥 평소처럼 호출하면 됨
wavs, sr = model.generate_voice_clone(
    text="안녕하세요! 오늘 정말 좋은 날이에요!",
    language="Korean",
    ref_audio=ref_audio,
    ref_text=ref_text,
)
sf.write("./audios/output_voice_clone.wav", wavs[0], sr)


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
